# 2장 · 흩어짐 재기 — 분산과 표준편차 📏

웹 챕터 **"흩어짐 재기 — 분산과 표준편차"** 와 짝을 이루는 Python 실습입니다.
편차의 합이 정말 0 이 되는지 확인하고, 분산·표준편차를 손으로 계산해
`np.var` / `np.std` 와 비교합니다. z점수로 서로 다른 시험 성적도 공평하게 비교해요.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요
from tutor import chat

chat('ch2')

## 0. 준비

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.default_rng(42)
print('준비 완료!')

## 1. 평균이 같아도: 두 반의 점수

웹 챕터의 두 반입니다. 평균은 둘 다 70점. 하지만 그래프는 전혀 다르죠.

In [ ]:
class_a = np.array([68, 69, 70, 71, 72])    # 다닥다닥
class_b = np.array([40, 55, 70, 85, 100])   # 들쭉날쭉

print('A반 평균:', class_a.mean(), ' / B반 평균:', class_b.mean())

x = np.arange(5)
plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, class_a, width=0.4, color='#818cf8', label='A반')
plt.bar(x + 0.2, class_b, width=0.4, color='#fbbf24', label='B반')
plt.axhline(70, color='#ef4444', linestyle='--', label='공통 평균 70')
plt.xticks(x, [f'학생{i+1}' for i in range(5)])
plt.ylabel('점수')
plt.title('평균은 똑같이 70점: 흩어짐은 전혀 다름')
plt.legend()
plt.show()

## 2. 편차의 합은 정말 0 일까?

편차 = 값 − 평균. 어떤 데이터든 편차를 전부 더하면 0 이 됩니다 (평균이 무게중심이라서).

In [ ]:
dev_b = class_b - class_b.mean()
print('B반 편차   :', dev_b)
print('편차의 합  :', dev_b.sum())            # 항상 0!

# 아무 데이터나 만들어도 0 인지 확인
random_data = rng.integers(0, 100, size=10)
print('아무 데이터 :', random_data)
print('편차의 합  :', (random_data - random_data.mean()).sum().round(10))

## 3. 분산·표준편차: 손계산 vs numpy

분산 = **편차 제곱의 평균**, 표준편차 = **√분산**. 공식 그대로 코드로 옮겨서
`np.var`, `np.std` 와 일치하는지 확인합니다.

In [ ]:
def variance_manual(data):
    data = np.asarray(data, dtype=float)
    deviations = data - data.mean()      # ① 편차
    squared = deviations ** 2            # ② 제곱 (부호 소멸)
    return squared.mean()                # ③ 평균

for name, cls in [('A반', class_a), ('B반', class_b)]:
    v = variance_manual(cls)
    print(f'{name}: 분산(직접) {v:6.1f} | np.var {np.var(cls):6.1f} | '
          f'표준편차 √분산 = {np.sqrt(v):5.2f} | np.std {np.std(cls):5.2f}')

> 참고: `np.var(x)` 의 기본값은 앞에서 배운 "n 으로 나누는" 분산입니다.
> 나중에 `ddof=1`(n−1 로 나누는 표본분산)이라는 옵션을 만나도 놀라지 마세요.
> 표본으로 모집단을 추정할 때 쓰는 보정인데, 8장 언저리에서 감이 잡힙니다.

## 4. z점수: 다른 시험끼리 공평하게 비교

수학 70점(평균 60, σ 10) vs 영어 80점(평균 75, σ 5). 어느 쪽을 더 잘 본 걸까요?

In [ ]:
def z_score(x, mean, std):
    return (x - mean) / std

z_math = z_score(70, 60, 10)
z_english = z_score(80, 75, 5)

print('수학 z점수 :', z_math)
print('영어 z점수 :', z_english)
print()
print('둘 다 z = 1.0 → 평균보다 표준편차 1개만큼 위, 완전히 같은 성취!')

## 5. σ 가 다른 두 분포: 그림으로 실감하기

평균이 같고 σ 만 다른 데이터 1000개씩을 겹쳐 그리면 σ 의 의미가 한눈에 보입니다.

🧪 **직접 실험**: σ 값(1.4, 21)을 바꿔서 다시 그려 보세요.

In [ ]:
narrow = rng.normal(70, 1.4, size=1000)   # A반 느낌 (σ 작음)
wide = rng.normal(70, 21, size=1000)      # B반 느낌 (σ 큼)

plt.figure(figsize=(8, 4))
plt.hist(wide, bins=40, color='#fbbf24', alpha=0.7, label='σ = 21 (넓게 퍼짐)')
plt.hist(narrow, bins=40, color='#818cf8', alpha=0.8, label='σ = 1.4 (좁게 모임)')
plt.axvline(70, color='#ef4444', linestyle='--', label='공통 평균 70')
plt.xlabel('점수')
plt.ylabel('개수')
plt.title('평균이 같아도 σ 가 다르면 전혀 다른 분포')
plt.legend()
plt.show()

## 정리

- 편차의 합은 언제나 0 → **제곱해서 평균** 낸 것이 분산, **√분산** 이 표준편차 σ.
- σ 는 "평균에서 대략 ±σ 만큼 흩어져 있다"로 읽습니다 (원래 단위!).
- **z점수 = (값 − 평균) ÷ σ**: 단위가 사라져 서로 다른 시험·특성끼리 비교 가능.
- AI 연결: 입력 특성을 z점수로 **표준화(feature scaling)** 하면 경사하강법 학습이 안정됩니다.

다음 노트북 `ch03_probability_basics.ipynb` 에서는 무대를 옮겨 **확률의 기초**를 다룹니다.